<a href="https://colab.research.google.com/github/LukeXboy/E-Commerce-Data-Analysis-Personal-Project/blob/main/01_data_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Liu Xing (Luke)'s Personal Project

In [1]:
#Load datasets
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
orders = pd.read_csv('/content/olist_orders_dataset.csv')
payments = pd.read_csv('/content/olist_order_payments_dataset.csv')
customers = pd.read_csv('/content/olist_customers_dataset.csv')
reviews = pd.read_csv('/content/olist_order_reviews_dataset.csv')
order_items = pd.read_csv('/content/olist_order_items_dataset.csv')
products = pd.read_csv('/content/olist_products_dataset.csv')
product_category_name_translation = pd.read_csv('/content/product_category_name_translation.csv')

##1. Data Quality

In [2]:
#1. Number of rows and columns
print(orders.shape)
#2. Column data types
print(orders.dtypes)
#3. Missing values for every column
print(orders.isnull().sum())
#4. Number of duplicated order_id values
print(orders['order_id'].duplicated().sum())
#5. Distribution/count of order_status
print(orders['order_status'].value_counts())
#6. Earliest and latest order purchase date
print(orders['order_purchase_timestamp'].min())
print(orders['order_purchase_timestamp'].max())

(99441, 8)
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
0
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64
2016-09-04 21:15:19
2018-10-17 17:30:18


In [3]:
#convert into datetime values
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(
        orders[col],
        errors='coerce'
    )

In [4]:
#check timestamp data type again
print(orders[date_cols].dtypes)
print(orders['order_purchase_timestamp'].min())
print(orders['order_purchase_timestamp'].max())

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object
2016-09-04 21:15:19
2018-10-17 17:30:18


In [5]:
#check number of missing values for each order_status
orders.groupby('order_status', as_index=False).agg(missing_delivery_count=('order_delivered_customer_date', lambda x: x.isnull().sum()))

,order_status,missing_delivery_count
0,approved,2
1,canceled,619
2,created,5
3,delivered,8
4,invoiced,314
5,processing,301
6,shipped,1107
7,unavailable,609


From the table above, we can see that missing delivery timestamps are largely consistent with non-delivered order statuses. However, 8 orders marked as delivered have a missing customer delivery timestamp and should be investigated as potential data-quality anomalies.

In [6]:
#investigation on the 8 inconsistent orders
inconsistent_orders = orders[(orders['order_status'] == 'delivered') & (orders['order_delivered_customer_date'].isnull())]
inconsistent_orders[[
    'order_id',
    'customer_id',
    'order_status',
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


In [7]:
print(f'number of orders that were approved before purchased: {len(pd.DataFrame(orders[orders['order_approved_at']<orders['order_purchase_timestamp']]))}')

print(f'number of orders that were delivered to customer before handed to carrier: {len(pd.DataFrame(orders[orders['order_delivered_customer_date']<orders['order_delivered_carrier_date']]))}')

number of orders that were approved before purchased: 0
number of orders that were delivered to customer before handed to carrier: 23


**Date consistency finding:** No orders were approved before purchase. However, 23 orders have a recorded customer-delivery timestamp earlier than the carrier handoff timestamp, indicating a small set of inconsistent lifecycle records that require investigation.

In [8]:
anomalous_orders=pd.DataFrame(orders[orders['order_delivered_customer_date']<orders['order_delivered_carrier_date']])
anomalous_orders[[
    'order_id',
    'customer_id',
    'order_status',
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]]
anomalous_orders['delivery_sequence_gap_hours']= anomalous_orders['order_delivered_customer_date']-anomalous_orders['order_delivered_carrier_date']
anomalous_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_sequence_gap_hours
6437,a1abeb653a4d4cd1e142ccb8c82cd069,5f50465da00b7fed5dd1239f4ecf6e2c,delivered,2017-07-20 11:20:52,2017-07-21 06:43:14,2017-07-28 16:57:58,2017-07-25 19:32:56,2017-08-14,-3 days +02:34:58
9553,383aa8b2724fe452d9ccd9934a8c628b,b1cb2f9d7a19480f3749e248db14d58f,delivered,2017-07-02 20:58:43,2017-07-02 21:10:20,2017-07-07 17:22:41,2017-07-06 14:27:51,2017-07-21,-2 days +21:05:10
13487,cb1134f9010d242e9515ad1c78ec0c39,2fd33ac77677bd214b1882868317eeed,delivered,2017-07-16 12:35:34,2017-07-18 06:03:50,2017-07-20 19:22:02,2017-07-19 14:13:28,2017-08-08,-2 days +18:51:26
14474,dceb62e8fa94b46006c9554fed743df0,2721900eb4e0f1cc2c836dd7bc1b1e11,delivered,2017-07-20 20:58:05,2017-07-22 11:45:11,2017-08-01 18:23:30,2017-07-26 18:09:10,2017-08-11,-7 days +23:45:40
19268,5f9d46795c3126674e52becb3a1a517f,79287bcaafdde5c793b996fc40bb7d9f,delivered,2017-07-18 11:48:20,2017-07-18 12:03:29,2017-07-20 23:03:42,2017-07-20 18:52:41,2017-07-31,-1 days +19:48:59


In [9]:
anomalous_orders = orders[
    orders['order_delivered_customer_date']
    < orders['order_delivered_carrier_date']
].copy()

anomalous_orders['delivery_sequence_gap_hours'] = (
    anomalous_orders['order_delivered_customer_date']
    - anomalous_orders['order_delivered_carrier_date']
).dt.total_seconds() / 3600
anomalous_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_sequence_gap_hours
6437,a1abeb653a4d4cd1e142ccb8c82cd069,5f50465da00b7fed5dd1239f4ecf6e2c,delivered,2017-07-20 11:20:52,2017-07-21 06:43:14,2017-07-28 16:57:58,2017-07-25 19:32:56,2017-08-14,-69.417222
9553,383aa8b2724fe452d9ccd9934a8c628b,b1cb2f9d7a19480f3749e248db14d58f,delivered,2017-07-02 20:58:43,2017-07-02 21:10:20,2017-07-07 17:22:41,2017-07-06 14:27:51,2017-07-21,-26.913889
13487,cb1134f9010d242e9515ad1c78ec0c39,2fd33ac77677bd214b1882868317eeed,delivered,2017-07-16 12:35:34,2017-07-18 06:03:50,2017-07-20 19:22:02,2017-07-19 14:13:28,2017-08-08,-29.142778
14474,dceb62e8fa94b46006c9554fed743df0,2721900eb4e0f1cc2c836dd7bc1b1e11,delivered,2017-07-20 20:58:05,2017-07-22 11:45:11,2017-08-01 18:23:30,2017-07-26 18:09:10,2017-08-11,-144.238889
19268,5f9d46795c3126674e52becb3a1a517f,79287bcaafdde5c793b996fc40bb7d9f,delivered,2017-07-18 11:48:20,2017-07-18 12:03:29,2017-07-20 23:03:42,2017-07-20 18:52:41,2017-07-31,-4.183611


**Lifecycle anomaly:** 23 orders have customer-delivery timestamps earlier than their carrier-handoff timestamps. The inconsistencies range from roughly hours to multiple days, so these records should be flagged and excluded from analyses that depend on delivery duration.

In [10]:
anomalous_orders['delivery_sequence_gap_hours'] = (
    anomalous_orders['order_delivered_customer_date']
    - anomalous_orders['order_delivered_carrier_date']
).dt.total_seconds() / 3600
print(f'count of anomalous orders: {len(anomalous_orders)}')
print(f'minimum gap in hours: {anomalous_orders["delivery_sequence_gap_hours"].min()}')
print(f'maximum gap in hours: {anomalous_orders["delivery_sequence_gap_hours"].max()}')
print(f'median gap in hours: {anomalous_orders["delivery_sequence_gap_hours"].median()}')

count of anomalous orders: 23
minimum gap in hours: -386.30805555555554
maximum gap in hours: -0.3883333333333333
median gap in hours: -39.864444444444445


23 orders have invalid delivery sequences. The most severe records show customer delivery recorded substantially before carrier handoff, so these rows will be excluded from delivery-duration analyses but retained for unrelated analyses such as revenue.

##2. Building analytical dataset

In [11]:
#merge all three datasets together
agg_payments = payments.groupby('order_id',as_index=False).agg(order_revenue=('payment_value','sum'))
merged = orders.merge(agg_payments,on='order_id',how='left')
final_df = merged.merge(customers,on='customer_id',how='left')[['order_id',
'customer_id',
'customer_unique_id',
'order_purchase_timestamp',
'order_status',
'order_revenue']]
final_df.head()

,order_id,customer_id,customer_unique_id,order_purchase_timestamp,order_status,order_revenue
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,2017-10-02 10:56:33,delivered,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,2018-07-24 20:41:37,delivered,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,2018-08-08 08:38:49,delivered,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,2017-11-18 19:28:06,delivered,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,2018-02-13 21:18:39,delivered,28.62


In [12]:
print(f'1. total number of rows: {len(final_df)}')
print(f'2. number of unique order_id values: {final_df["order_id"].nunique()}')
print(f'3. number of duplicated order_id values: {final_df["order_id"].duplicated().sum()}')
print(f'4. number of missing order_revenue values: {final_df["order_revenue"].isnull().sum()}')
print(f'5. number of missing customer_unique_id values: {final_df["customer_unique_id"].isnull().sum()}')

1. total number of rows: 99441
2. number of unique order_id values: 99441
3. number of duplicated order_id values: 0
4. number of missing order_revenue values: 1
5. number of missing customer_unique_id values: 0


In [13]:
#find the missing order revenue
final_df[final_df['order_revenue'].isnull()].copy()[['order_id',
'customer_id',
'customer_unique_id',
'order_purchase_timestamp',
'order_status',
'order_revenue']]

,order_id,customer_id,customer_unique_id,order_purchase_timestamp,order_status,order_revenue
30710,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,830d5b7aaa3b6f1e9ad63703bec97d23,2016-09-15 12:16:38,delivered,NaN


In [14]:
#flagged the missing value
final_df['missing_revenue_flag'] = final_df['order_revenue'].isna()

In [15]:
payments[payments['order_id']=='bfbd0f9bdef84302105ad712db648a6c']

,order_id,payment_sequential,payment_type,payment_installments,payment_value


In [16]:
#find the order from order items
order_items[order_items['order_id']=='bfbd0f9bdef84302105ad712db648a6c']

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
84389,bfbd0f9bdef84302105ad712db648a6c,1,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83
84390,bfbd0f9bdef84302105ad712db648a6c,2,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83
84391,bfbd0f9bdef84302105ad712db648a6c,3,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83


In [17]:
print(f'The actual revenue of the missing order is 3 x (44.99 + 2.83) = 143.46')
final_df.loc[
    final_df['order_id'] == 'bfbd0f9bdef84302105ad712db648a6c',
    'order_revenue'
] = 143.46

The actual revenue of the missing order is 3 x (44.99 + 2.83) = 143.46


In [18]:
reviews_agg = reviews.groupby('order_id',as_index=False).agg(
    avg_review_score=('review_score','mean'),
    review_count=('review_id','count'))
final_df_reviewed = final_df.merge(reviews_agg,on='order_id',how='left')

In [19]:
final_df_reviewed['order_id'].duplicated().sum()

np.int64(0)

In [20]:
final_df_reviewed['avg_review_score'].isnull().sum()

np.int64(768)

In [21]:
len(final_df_reviewed)

99441

In [22]:
missing_reviews_df = final_df_reviewed.groupby('order_status',as_index=False).agg(
    total_orders = ('order_id','count'),
    missing_review_count=('review_count', lambda x: x.isnull().sum()))
missing_reviews_df['missing_review_pct']= missing_reviews_df['missing_review_count']*100/missing_reviews_df['total_orders']
missing_reviews_df.sort_values(by='missing_review_pct',ascending=False)

,order_status,total_orders,missing_review_count,missing_review_pct
2,created,5,2,40.000000
6,shipped,1107,75,6.775068
1,canceled,625,20,3.200000
7,unavailable,609,14,2.298851
5,processing,301,6,1.993355
4,invoiced,314,5,1.592357
3,delivered,96478,646,0.669583
0,approved,2,0,0.000000


Review missingness varies by order status. Delivered orders have a very low missing-review rate (~0.67%), while non-delivered statuses such as shipped and canceled show higher missingness. Therefore, missing review scores will be retained as null rather than imputed.

##3. KPI analysis

In [23]:
monthly_KPI = final_df_reviewed[final_df_reviewed['order_status']=='delivered'].copy()
monthly_KPI['month'] = (
    monthly_KPI['order_purchase_timestamp']
    .dt.to_period('M')
)

monthly_KPI = monthly_KPI.groupby('month', as_index=False).agg(
    revenue=('order_revenue', 'sum'),
    unique_customers=('customer_unique_id', 'nunique'),
    orders=('order_id', 'nunique'),
    avg_review_score=('avg_review_score', 'mean')
)

monthly_KPI['avg_order_value'] = monthly_KPI['revenue'] / monthly_KPI['orders']
monthly_KPI

,month,revenue,unique_customers,orders,avg_review_score,avg_order_value
0,2016-09,143.46,1,1,1.000000,143.460000
1,2016-10,46566.71,262,265,4.011450,175.723434
2,2016-12,19.62,1,1,5.000000,19.620000
3,2017-01,127545.67,718,750,4.199055,170.060893
4,2017-02,271298.65,1630,1653,4.202678,164.125015
5,2017-03,414369.39,2508,2546,4.187376,162.753099
6,2017-04,390952.18,2274,2303,4.137555,169.757785
7,2017-05,567066.73,3479,3546,4.238488,159.917296
8,2017-06,490225.60,3076,3135,4.222919,156.371802
9,2017-07,566403.93,3802,3872,4.258416,146.282007


From 2017 onward, I see an overall upward trend in revenue, customers, and order volume, suggesting growth was largely volume-driven. AOV fluctuated but remained relatively stable compared with the increase in transaction volume, so higher AOV does not appear to be the main long-term growth driver. There is also a noticeable jump in activity around late 2017, which I would investigate further for seasonality or promotional effects. Review scores were generally stable around 4 or above, although they weakened around late 2017 to early 2018 before recovering. I would avoid interpreting the extreme 2016 values because those months contain very few orders.

In [24]:
#analyzing 2017-10 to 2017-11 revenue surge
revenue_decomp = final_df_reviewed[(final_df_reviewed['order_status']=='delivered')&(final_df_reviewed['order_purchase_timestamp']>='2017-10-01')&(final_df_reviewed['order_purchase_timestamp']<'2017-12-01')].copy()
revenue_decomp['month']=revenue_decomp['order_purchase_timestamp'].dt.to_period('M')
revenue_decomp = revenue_decomp.groupby('month',as_index=False).agg(
    revenue=('order_revenue','sum'),
    unique_customers=('customer_unique_id','nunique'),
    orders=('order_id','nunique'))
revenue_decomp['orders_per_customer']=revenue_decomp['orders']/revenue_decomp['unique_customers']
revenue_decomp['avg_order_value']=revenue_decomp['revenue']/revenue_decomp['orders']
revenue_decomp

,month,revenue,unique_customers,orders,orders_per_customer,avg_order_value
0,2017-10,751140.27,4417,4478,1.013810,167.740123
1,2017-11,1153528.05,7183,7289,1.014757,158.256009


Revenue increased by about **54%** from October to November. The main driver was volume: unique customers and orders both increased by roughly **63%**, while orders per customer remained almost unchanged. AOV actually decreased by about **6%**, so the revenue increase was not driven by customers spending more per order. I would therefore investigate what caused the large increase in customer/order volume, such as seasonality, promotions, product categories, seller activity, or regional demand.

In [25]:
from google.colab import files
#final_df_reviewed.to_csv('final_df_reviewed.csv', index=False)
#files.download('final_df_reviewed.csv')

##4. Deeper investigation

In [26]:
#merge multiple datasets to analyze product-category contribution to the Nov 2017 volume surge
df = final_df_reviewed.merge(order_items,on='order_id',how='left')
df = df.merge(products,on='product_id',how='left')
df = df.merge(product_category_name_translation,on='product_category_name',how='left')
filtered_df = df[(df['order_purchase_timestamp']>='2017-10-01')&
 (df['order_purchase_timestamp']<'2017-12-01')&(df['order_status']=='delivered')].copy()
filtered_df['month']=filtered_df['order_purchase_timestamp'].dt.to_period('M')
filtered_df = filtered_df.groupby(['month','product_category_name_english'],as_index=False).agg(
    orders=('order_id','nunique'),
    items_sold=('order_item_id','count'),
    item_revenue=('price','sum'))
filtered_df = filtered_df.sort_values(
    by=['month', 'item_revenue'],
    ascending=[True, False])
filtered_df

,month,product_category_name_english,orders,items_sold,item_revenue
62,2017-10,watches_gifts,282,308,64874.63
57,2017-10,sports_leisure,417,462,48568.98
6,2017-10,bed_bath_table,442,542,46007.70
18,2017-10,cool_stuff,242,254,44348.23
13,2017-10,computers_accessories,253,309,42009.38
...,...,...,...,...,...
86,2017-11,dvds_blu_ray,3,3,79.79
65,2017-11,art,1,1,75.00
88,2017-11,fashio_female_clothing,2,2,59.98
92,2017-11,fashion_sport,1,1,59.90


In [27]:
oct_revenue = filtered_df[filtered_df['month']=='2017-10'].groupby('product_category_name_english',as_index=False).agg(oct_revenue=('item_revenue','sum'))
nov_revenue = filtered_df[filtered_df['month']=='2017-11'].groupby('product_category_name_english',as_index=False).agg(nov_revenue=('item_revenue','sum'))
category_contribution = oct_revenue.merge(nov_revenue,on='product_category_name_english',how='outer')
category_contribution[['oct_revenue', 'nov_revenue']] = (
    category_contribution[['oct_revenue', 'nov_revenue']]
    .fillna(0))
category_contribution['revenue_change']=category_contribution['nov_revenue']-category_contribution['oct_revenue']
category_contribution['growth_pct']=np.where(
    category_contribution['oct_revenue'] > 0,
    category_contribution['revenue_change']
    * 100
    / category_contribution['oct_revenue'],
    np.nan)
category_contribution = category_contribution.sort_values(by='revenue_change',ascending=False)
category_contribution.head(10)

,product_category_name_english,oct_revenue,nov_revenue,revenue_change,growth_pct
6,bed_bath_table,46007.70,87957.63,41949.93,91.180237
41,health_beauty,40698.99,78274.40,37575.41,92.325166
37,furniture_decor,30009.74,62091.27,32081.53,106.903725
66,watches_gifts,64874.63,95292.34,30417.71,46.886911
65,toys,33324.42,62611.26,29286.84,87.884020
14,computers_accessories,42009.38,69676.32,27666.94,65.858958
40,garden_tools,25798.75,44275.18,18476.43,71.617540
47,housewares,15883.68,33872.42,17988.74,113.252974
57,perfumery,16877.82,31612.13,14734.31,87.299841
61,sports_leisure,48568.98,62686.24,14117.26,29.066412


Bed bath table contributed the most in November revenue increase, but we can not simply say it is the cause of the November revenue surge since there could be many other factors: seasonal effect, promotions/discounts, etc.

In [28]:
detail_df = df[
    (df['order_purchase_timestamp'] >= '2017-10-01') &
    (df['order_purchase_timestamp'] < '2017-12-01') &
    (df['order_status'] == 'delivered')].copy()

detail_df['month'] = (
    detail_df['order_purchase_timestamp']
    .dt.to_period('M'))

price_check = (
    detail_df.groupby('month', as_index=False)
    .agg(
        items_sold=('order_item_id', 'count'),
        item_revenue=('price', 'sum'),
        orders=('order_id', 'nunique')))

price_check['avg_item_price'] = (
    price_check['item_revenue']
    / price_check['items_sold'])

price_check['items_per_order'] = (
    price_check['items_sold']
    / price_check['orders'])
price_check

,month,items_sold,item_revenue,orders,avg_item_price,items_per_order
0,2017-10,5214,648247.65,4478,124.328280,1.164359
1,2017-11,8475,987765.37,7289,116.550486,1.162711


The November revenue surge was primarily volume-driven. Order count and item volume increased substantially, while items per order remained nearly unchanged and average item price declined. That means growth came from more transactions rather than larger baskets or higher prices.

In [29]:
state_df = final_df_reviewed.merge(
    customers[['customer_id', 'customer_state']],
    on='customer_id',
    how='left'
)

state_df = state_df[
    (state_df['order_status'] == 'delivered') &
    (state_df['order_purchase_timestamp'] >= '2017-10-01') &
    (state_df['order_purchase_timestamp'] < '2017-12-01')
].copy()

state_df['month'] = (
    state_df['order_purchase_timestamp']
    .dt.to_period('M')
)

state_summary = (
    state_df
    .groupby(['month', 'customer_state'], as_index=False)
    .agg(
        unique_customers=('customer_unique_id', 'nunique'),
        orders=('order_id', 'nunique'),
        revenue=('order_revenue', 'sum')
    )
)
oct_state_revenue = state_summary[state_summary['month']=='2017-10'][['customer_state','revenue']]
nov_state_revenue = state_summary[state_summary['month']=='2017-11'][['customer_state','revenue']]
state_revenue=oct_state_revenue.merge(nov_state_revenue,on='customer_state',how='outer',suffixes=('-oct','-nov'))
state_revenue[['revenue-oct', 'revenue-nov']] = (
    state_revenue[['revenue-oct', 'revenue-nov']]
    .fillna(0))
state_revenue['revenue_change']=state_revenue['revenue-nov']-state_revenue['revenue-oct']
state_revenue.sort_values(by='revenue_change',ascending=False)

,customer_state,revenue-oct,revenue-nov,revenue_change
25,SP,249924.09,401027.72,151103.63
18,RJ,110577.92,172234.57,61656.65
10,MG,93049.17,154131.98,61082.81
22,RS,43173.06,67262.74,24089.68
23,SC,24773.77,44269.59,19495.82
17,PR,37404.60,54237.96,16833.36
7,ES,12667.55,28402.83,15735.28
6,DF,14731.50,30196.02,15464.52
4,BA,27614.20,35449.55,7835.35
5,CE,14338.60,22128.37,7789.77


The November revenue surge was geographically concentrated. São Paulo contributed the largest absolute increase, followed by Rio de Janeiro and Minas Gerais. Together, these three states accounted for roughly **68%** of the overall month-over-month revenue increase, suggesting that a substantial portion of the growth came from Brazil’s largest customer markets rather than being evenly distributed across all states.

In [30]:
from enum import unique
selected_states = state_summary.loc[state_summary['customer_state'].isin(['SP', 'RJ', 'MG', 'RS'])]
selected_states_agg=selected_states.groupby(['month','customer_state'],as_index=False).agg(
    unique_customers=('unique_customers','sum'),
    orders=('orders','sum'),
    revenue=('revenue','sum')
)
selected_states_agg['orders_per_customer']=selected_states_agg['orders']/selected_states_agg['unique_customers']
selected_states_agg['revenue_per_customer']=selected_states_agg['revenue']/selected_states_agg['unique_customers']
selected_states_agg['avg_order_value']=selected_states_agg['revenue']/selected_states_agg['orders']
selected_states_agg

,month,customer_state,unique_customers,orders,revenue,orders_per_customer,revenue_per_customer,avg_order_value
0,2017-10,MG,533,542,93049.17,1.016886,174.576304,171.677435
1,2017-10,RJ,637,654,110577.92,1.026688,173.591711,169.079388
2,2017-10,RS,242,244,43173.06,1.008264,178.401074,176.938770
3,2017-10,SP,1703,1723,249924.09,1.011744,146.755191,145.051706
4,2017-11,MG,914,922,154131.98,1.008753,168.634551,167.171345
5,2017-11,RJ,996,1012,172234.57,1.016064,172.926275,170.192263
6,2017-11,RS,405,409,67262.74,1.009877,166.080840,164.456577
7,2017-11,SP,2844,2899,401027.72,1.019339,141.008340,138.333122


The November growth in these major states was primarily driven by many more customers entering and placing orders, not by existing customers ordering more frequently or spending more per order.

In [31]:
customer_history = final_df_reviewed.merge(
    customers[['customer_id', 'customer_state']],
    on='customer_id',
    how='left'
)
customer_history = customer_history[
    customer_history['order_status'] == 'delivered'
].copy()

customer_history['month'] = (
    customer_history['order_purchase_timestamp']
    .dt.to_period('M')
)
before_nov_df = customer_history[customer_history['month'] < '2017-11'].copy()
nov_df_selected_states = customer_history[(customer_history['month'] == '2017-11')&(customer_history['customer_state'].isin(['SP', 'RJ', 'MG', 'RS']))].copy()

# Identify unique customer IDs from before November
past_customers = set(before_nov_df['customer_unique_id'].unique())

# Vectorized mapping using np.where for maximum efficiency
nov_df_selected_states['customer_label'] = np.where(
    nov_df_selected_states['customer_unique_id'].isin(past_customers),
    'returning',
    'new'
)
nov_df_selected_states

,order_id,customer_id,customer_unique_id,order_purchase_timestamp,order_status,order_revenue,missing_revenue_flag,avg_review_score,review_count,customer_state,month,customer_label
18,85ce859fd6dc634de8d2f1e290444043,059f7fc5719c7da6cbafe370971a8d70,d0ff1a7468fcc46b8fc658ab35d2a12c,2017-11-21 00:03:41,delivered,29.75,False,5.0,1.0,SP,2017-11,new
41,6ea2f835b4556291ffdc53fa0b3b95e8,c7340080e394356141681bd4c9b8fe31,3e4fd73f1e86b135b9b121d6abbe9597,2017-11-24 21:27:48,delivered,356.12,False,1.0,1.0,SP,2017-11,new
60,68873cf91053cd11e6b49a766db5af1a,4632eb5a8f175f6fe020520ae0c678f3,6da92ae920ab16fc4eceb8fcd7bd43ce,2017-11-30 22:02:15,delivered,91.66,False,4.0,1.0,SP,2017-11,new
62,8f06cc6465925031568537b815f1198d,9916715c2ab6ee1710c9c32f0a534ad2,bf0303939d54b8df5da4762bbaab1955,2017-11-15 11:31:41,delivered,317.34,False,5.0,1.0,RJ,2017-11,new
97,6a0a8bfbbe700284feb0845d95e0867f,68451b39b1314302c08c65a29f1140fc,781ae350edb16842380e81d7c7feb431,2017-11-22 11:32:22,delivered,101.74,False,1.0,1.0,RJ,2017-11,new
...,...,...,...,...,...,...,...,...,...,...,...,...
99388,c0524fb1b4c905d054adbddaffa2380c,92e8f9754238b9697d9dcbe02c20fc70,3f47ce7f07f450a63c6a1a7460074d75,2017-11-24 16:44:08,delivered,97.71,False,2.0,1.0,RS,2017-11,new
99393,29b199897e15bd5a4b501cf5eee8b948,530575e87308897d871c2008d28713dc,3b6572635fb53a6846bacb6947a91e93,2017-11-19 18:00:41,delivered,145.14,False,5.0,1.0,MG,2017-11,new
99412,4146f35ac7a7ef4e39fe344e563c1e3b,e0eac226ceae49858de89e8a5c1ba1ec,7d414b6a1cf56a6bc5035f9eb436b15d,2017-11-26 19:42:22,delivered,141.40,False,5.0,1.0,SP,2017-11,new
99424,6ec4642f9993cc34f826cfb9068e5a2f,f00bb12620b7b232233cace1b229db18,cf82c9581d66ad1de6b0fe919188bdd6,2017-11-21 20:18:03,delivered,319.25,False,4.0,1.0,MG,2017-11,new


In [35]:
new_customers = nov_df_selected_states[nov_df_selected_states['customer_label']=='new'].groupby('customer_state').agg(
    new_customers=('customer_unique_id','nunique')
)
returning_customers = nov_df_selected_states[nov_df_selected_states['customer_label']=='returning'].groupby('customer_state').agg(
    returning_customers=('customer_unique_id','nunique')
)
customer_summary = new_customers.merge(returning_customers,on='customer_state',how='outer')
customer_summary['total_customers']=customer_summary['new_customers']+customer_summary['returning_customers']
customer_summary['new_customers_pct']=customer_summary['new_customers']*100/customer_summary['total_customers']
customer_summary

,new_customers,returning_customers,total_customers,new_customers_pct
customer_state,,,,
MG,893,21,914,97.702407
RJ,979,17,996,98.293173
RS,399,6,405,98.518519
SP,2803,41,2844,98.558368


Across SP, RJ, MG, and RS, approximately **98%** of November 2017 customers had no prior delivered order in the observed dataset. Combined with stable order frequency and lower AOV, this suggests the November revenue surge was primarily associated with a large influx of new customers rather than increased spending by existing customers.